# Layer 2 — Code Risk Checker

Flags data-journalism risk patterns in Python and R analysis scripts.  
**Does not run the code.** Static analysis only — no execution, no data required.

Flag taxonomy: `data/documentation/LAYER2_FLAGS.md`

---

## Setup

In [1]:
import ast
import re
import textwrap
from dataclasses import dataclass, field
from pathlib import Path
from IPython.display import HTML, display

## CodeFlag — core data type

In [2]:
@dataclass
class CodeFlag:
    line: int          # 1-indexed line number
    col: int           # 0-indexed column offset
    end_line: int      # last line of flagged span
    code: str          # exact source text of the flagged line(s)
    flag_type: str
    priority: str      # High / Medium
    reason: str
    language: str      # python / r

# Color per flag type — consistent with Layer 1 palette where types overlap
FLAG_COLORS = {
    "no_shape_check":              "#ff6b6b",   # red
    "no_na_check":                 "#ff922b",   # orange
    "no_dtype_check":              "#ffd43b",   # yellow
    "zip_as_numeric":              "#f03e3e",   # dark red
    "encoding_not_set":            "#a9e34b",   # green
    "excel_date_risk":             "#63e6be",   # teal
    "no_value_range_check":        "#74c0fc",   # blue
    "no_category_check":           "#4dabf7",   # mid blue
    "total_row_risk":              "#ff6b6b",   # red
    "magic_number":                "#dee2e6",   # grey
    "sentinel_value_risk":         "#ff922b",   # orange
    "no_join_count_check":         "#f03e3e",   # dark red
    "join_on_string":              "#ffa8a8",   # pink
    "no_unmatched_check":          "#ff6b6b",   # red
    "hardcoded_threshold":         "#ffd43b",   # yellow
    "percentage_without_base":     "#ff922b",   # orange
    "small_denominator_risk":      "#f03e3e",   # dark red
    "mean_without_median":         "#74c0fc",   # blue
    "no_null_before_aggregation":  "#ff6b6b",   # red
    "pct_change_without_base_note":"#a9e34b",   # green
    "geocoding_unverified":        "#ff922b",   # orange
    "projection_not_set":          "#f03e3e",   # dark red
    "hardcoded_geo_count":         "#ffd43b",   # yellow
}

PRIORITY_ORDER = {"High": 0, "Medium": 1}

## Python flagging — AST + regex

In [3]:
# ── Python helpers ──────────────────────────────────────────────────────────

def _src_line(source_lines: list[str], lineno: int) -> str:
    """Return the source line (1-indexed), rstripped."""
    return source_lines[lineno - 1].rstrip() if 0 < lineno <= len(source_lines) else ""


def _code_only(line: str) -> str:
    """Strip inline comment — avoids comments with flag keywords triggering false negatives."""
    idx = line.find("#")
    return line[:idx] if idx != -1 else line


def _window(source_lines: list[str], lineno: int, before: int = 5, after: int = 5) -> list[str]:
    """Lines in a window around lineno (1-indexed), excluding lineno itself."""
    start = max(0, lineno - 1 - before)
    end   = min(len(source_lines), lineno + after)
    return source_lines[start: lineno - 1] + source_lines[lineno: end]


def _has_nearby_call(source_lines: list[str], lineno: int, methods: list[str],
                     before: int = 5, after: int = 5) -> bool:
    """True if any method name appears in the window around lineno (comments stripped)."""
    joined = " ".join(_code_only(l) for l in _window(source_lines, lineno, before, after))
    return any(m in joined for m in methods)


# Sentinel value patterns — negative/large codes masquerading as missing data
_SENTINEL_RE = re.compile(r"!=\s*-(?:99+|999+)|!=\s*(?:9999|99999)\b")

# ZIP column name patterns
_ZIP_PATTERNS = re.compile(r"\bzip(?:_?code)?s?\b", re.IGNORECASE)

# pct_change call
_PCT_CHANGE = re.compile(r"\.pct_change\s*\(")


class PythonFlagger(ast.NodeVisitor):
    """Walk the Python AST and collect CodeFlags."""

    def __init__(self, source: str):
        self.source_lines = source.splitlines()
        self.flags: list[CodeFlag] = []
        self._read_csv_lines: list[int] = []
        self._merge_lines: list[int] = []
        self._agg_lines: list[int] = []
        self._merge_how: dict[int, str] = {}

    def _flag(self, node, flag_type, priority, reason):
        lineno = node.lineno
        end_line = getattr(node, "end_lineno", lineno)
        self.flags.append(CodeFlag(
            line=lineno, col=node.col_offset, end_line=end_line,
            code=_src_line(self.source_lines, lineno),
            flag_type=flag_type, priority=priority, reason=reason, language="python",
        ))

    def _get_func_name(self, node: ast.Call) -> str | None:
        if isinstance(node.func, ast.Attribute): return node.func.attr
        if isinstance(node.func, ast.Name): return node.func.id
        return None

    # ── Load / IO ──────────────────────────────────────────────────────────

    def visit_Call(self, node: ast.Call):
        fn = self._get_func_name(node)

        if fn in ("read_csv", "read_excel", "read_table", "read_parquet"):
            self._read_csv_lines.append(node.lineno)
            if fn in ("read_csv", "read_table"):
                if not any(kw.arg == "encoding" for kw in node.keywords):
                    self._flag(node, "encoding_not_set", "Medium",
                               f"{fn}() with no encoding= argument")
            if fn == "read_excel":
                if not any(kw.arg == "dtype" for kw in node.keywords):
                    self._flag(node, "excel_date_risk", "Medium",
                               "read_excel() with no dtype= — date columns may parse incorrectly")

        if fn in ("merge", "join"):
            self._merge_lines.append(node.lineno)
            how = "inner"
            for kw in node.keywords:
                if kw.arg == "how" and isinstance(kw.value, ast.Constant):
                    how = kw.value.value
            self._merge_how[node.lineno] = how
            for kw in node.keywords:
                if kw.arg == "on" and isinstance(kw.value, ast.Constant) and isinstance(kw.value.value, str):
                    col = kw.value.value
                    if not re.search(r"(?:id|fips|code|num|key|_id)$", col, re.IGNORECASE):
                        self._flag(node, "join_on_string", "Medium",
                                   f"Merge key '{col}' looks like a string column — verify uniqueness")

        if fn in ("sum", "mean", "count", "median", "std", "var"):
            self._agg_lines.append(node.lineno)

        if "geocod" in (fn or ""):
            if not _has_nearby_call(self.source_lines, node.lineno,
                                    ["match_rate", "len(", "shape", "count"]):
                self._flag(node, "geocoding_unverified", "High",
                           "Geocoding call with no match-rate check nearby")

        if fn in ("sjoin", "sjoin_nearest"):
            if not _has_nearby_call(self.source_lines, node.lineno,
                                    ["crs", "epsg", "set_crs", "to_crs"]):
                self._flag(node, "projection_not_set", "High",
                           "Spatial join with no CRS check nearby")

        self.generic_visit(node)

    # ── Column / value ─────────────────────────────────────────────────────

    def visit_Assign(self, node: ast.Assign):
        """Detect ZIP cast to numeric via astype()."""
        if isinstance(node.value, ast.Call):
            fn = self._get_func_name(node.value)
            if fn == "astype" and node.value.args:
                arg = node.value.args[0]
                is_numeric = (
                    (isinstance(arg, ast.Name) and arg.id in ("int", "float", "int64", "float64")) or
                    (isinstance(arg, ast.Constant) and arg.value in ("int", "float", "int64", "float64"))
                )
                if is_numeric and _ZIP_PATTERNS.search(ast.unparse(node)):
                    self._flag(node, "zip_as_numeric", "High",
                               "ZIP code column cast to numeric — leading zeros will be lost")
        self.generic_visit(node)

    def visit_Compare(self, node: ast.Compare):
        """Detect hardcoded stat threshold (0.05 as a Constant)."""
        for comparator in node.comparators:
            if isinstance(comparator, ast.Constant) and comparator.value == 0.05:
                self._flag(node, "hardcoded_threshold", "High",
                           "Hardcoded p < 0.05 — document the alpha choice")
        self.generic_visit(node)

    # ── Post-checks ────────────────────────────────────────────────────────

    def run_post_checks(self):
        self._check_loads()
        self._check_merges()
        self._check_aggregations()
        self._check_regex_passes()

    def _check_loads(self):
        """Per-load checks — window stops at the next load line to avoid cross-load pollution."""
        all_load_lines = set(self._read_csv_lines)
        for lineno in self._read_csv_lines:
            after_lines = []
            for offset in range(1, 9):
                nxt = lineno + offset
                if nxt in all_load_lines:
                    break
                if nxt <= len(self.source_lines):
                    after_lines.append(self.source_lines[nxt - 1])
            before_lines = self.source_lines[max(0, lineno - 4): lineno - 1]
            # Strip comments — flag-name keywords in comments cause false negatives
            block = " ".join(_code_only(l) for l in before_lines + after_lines)
            lc = _src_line(self.source_lines, lineno)

            if not any(m in block for m in ["len(", "shape", "nrow", "count"]):
                self.flags.append(CodeFlag(line=lineno, col=0, end_line=lineno, code=lc,
                    flag_type="no_shape_check", priority="High",
                    reason="Data loaded — no row count check nearby (len/shape)",
                    language="python"))

            if not any(m in block for m in ["isna", "isnull", "notna", "dropna", "fillna"]):
                self.flags.append(CodeFlag(line=lineno, col=0, end_line=lineno, code=lc,
                    flag_type="no_na_check", priority="High",
                    reason="Data loaded — no NA check nearby (isna/dropna)",
                    language="python"))

            if not any(m in block for m in ["dtypes", "info(", ".dtype"]):
                self.flags.append(CodeFlag(line=lineno, col=0, end_line=lineno, code=lc,
                    flag_type="no_dtype_check", priority="Medium",
                    reason="Data loaded — no dtype check nearby (.dtypes/.info())",
                    language="python"))

    def _check_merges(self):
        for lineno in self._merge_lines:
            if not _has_nearby_call(self.source_lines, lineno,
                                    ["len(", "shape", "print"], before=3, after=3):
                self.flags.append(CodeFlag(line=lineno, col=0, end_line=lineno,
                    code=_src_line(self.source_lines, lineno),
                    flag_type="no_join_count_check", priority="High",
                    reason="Merge with no row count check before or after",
                    language="python"))
            how = self._merge_how.get(lineno, "inner")
            if how in ("left", "right", "outer"):
                if not _has_nearby_call(self.source_lines, lineno,
                                        ["isin", "indicator", "anti", "~"],
                                        before=4, after=8):
                    self.flags.append(CodeFlag(line=lineno, col=0, end_line=lineno,
                        code=_src_line(self.source_lines, lineno),
                        flag_type="no_unmatched_check", priority="High",
                        reason=f"{how} join — no check for unmatched rows",
                        language="python"))

    def _check_aggregations(self):
        for lineno in self._agg_lines:
            line = self.source_lines[lineno - 1]
            # Skip: null handling is chained on the same line
            if any(m in _code_only(line) for m in
                   ["dropna", "fillna", "notna", "skipna", "isna", "isnull"]):
                continue
            if not _has_nearby_call(self.source_lines, lineno,
                                    ["dropna", "fillna", "isna", "notna", "isnull"],
                                    before=5, after=2):
                self.flags.append(CodeFlag(line=lineno, col=0, end_line=lineno,
                    code=line.rstrip(),
                    flag_type="no_null_before_aggregation", priority="High",
                    reason="Aggregation with no prior null handling",
                    language="python"))

    def _check_regex_passes(self):
        """Line-level checks easier as text than AST."""
        for i, line in enumerate(self.source_lines, start=1):
            s = line.strip()
            c = _code_only(line)  # code without comment
            if not s or s.startswith("#"):
                continue

            # total_row_risk — equality check on "Total"/"total"
            if re.search(r'["\'](?:total|Total)["\']', c) and re.search(r"!=|==", c):
                self.flags.append(CodeFlag(line=i, col=0, end_line=i, code=s,
                    flag_type="total_row_risk", priority="High",
                    reason='"Total" row detected — verify exclusion from aggregation',
                    language="python"))

            # sentinel_value_risk — != -99 / != -999 as a literal
            if _SENTINEL_RE.search(c):
                self.flags.append(CodeFlag(line=i, col=0, end_line=i, code=s,
                    flag_type="sentinel_value_risk", priority="High",
                    reason="Sentinel value filter — verify this is not actual missing data",
                    language="python"))

            # percentage_without_base
            if re.search(r"[/\*]\s*100", c) and re.search(r"\bpct\b|percent|rate", c, re.IGNORECASE):
                nearby = self.source_lines[max(0, i - 3): i + 3]
                if not any(re.search(r"n=|n =", l) or "print" in l for l in nearby):
                    self.flags.append(CodeFlag(line=i, col=0, end_line=i, code=s,
                        flag_type="percentage_without_base", priority="High",
                        reason="Percentage calculated — denominator not printed nearby",
                        language="python"))

            # mean_without_median
            if ".mean()" in c:
                nearby = self.source_lines[max(0, i - 4): i + 4]
                if not any(".median()" in l for l in nearby):
                    self.flags.append(CodeFlag(line=i, col=0, end_line=i, code=s,
                        flag_type="mean_without_median", priority="Medium",
                        reason="mean() used — no median() nearby (check for outliers)",
                        language="python"))

            # no_value_range_check
            if re.search(r"\.(mean|sum)\(", c):
                nearby = self.source_lines[max(0, i - 6): i + 6]
                if not any(re.search(r"\.(min|max|describe)\(", l) for l in nearby):
                    self.flags.append(CodeFlag(line=i, col=0, end_line=i, code=s,
                        flag_type="no_value_range_check", priority="Medium",
                        reason="Aggregation with no min/max range check nearby",
                        language="python"))

            # no_category_check
            if ".groupby(" in c:
                nearby = self.source_lines[max(0, i - 6): i + 6]
                if not any("value_counts" in l for l in nearby):
                    self.flags.append(CodeFlag(line=i, col=0, end_line=i, code=s,
                        flag_type="no_category_check", priority="Medium",
                        reason="groupby() with no value_counts() to verify categories",
                        language="python"))

            # pct_change_without_base_note
            if _PCT_CHANGE.search(c):
                nearby = self.source_lines[max(0, i - 3): i + 3]
                if not any("#" in l for l in nearby):
                    self.flags.append(CodeFlag(line=i, col=0, end_line=i, code=s,
                        flag_type="pct_change_without_base_note", priority="Medium",
                        reason="pct_change() with no comment explaining base period",
                        language="python"))

            # magic_number
            m = re.search(r"[><!]=?\s*(\d+\.\d+)\b", c)
            if m:
                val = float(m.group(1))
                if val not in (0.0, 0.5, 1.0, 100.0, 0.05):
                    self.flags.append(CodeFlag(line=i, col=0, end_line=i, code=s,
                        flag_type="magic_number", priority="Medium",
                        reason=f"Unexplained threshold {m.group(1)} — add a comment",
                        language="python"))


## R flagging — regex-based (no AST)

In [4]:
# R patterns — line-by-line regex
# R has no standard AST library accessible from Python, so we use heuristic regexes.

_R_LOAD_FUNCS = re.compile(r"\b(?:read\.csv|read_csv|read\.table|fread|readRDS|load)\s*\(")
_R_JOIN_FUNCS = re.compile(r"\b(?:left_join|right_join|full_join|inner_join|merge)\s*\(")
_R_AGG_FUNCS  = re.compile(r"\b(?:sum|mean|count|n\(|summarise|summarize)\s*\(")
_R_GROUP_BY   = re.compile(r"\bgroup_by\s*\(")


def _r_has_nearby(lines: list[str], lineno: int, patterns: list[str], window: int = 6) -> bool:
    start = max(0, lineno - 1 - window)
    end   = min(len(lines), lineno + window)
    block = " ".join(lines[start:end])
    return any(p in block for p in patterns)


def flag_r(source: str) -> list[CodeFlag]:
    """Flag risk patterns in R source code."""
    lines = source.splitlines()
    flags: list[CodeFlag] = []

    load_lines: list[int] = []
    merge_lines: list[int] = []
    merge_outer: list[int] = []

    def _flag(lineno, flag_type, priority, reason):
        flags.append(CodeFlag(
            line=lineno, col=0, end_line=lineno,
            code=lines[lineno-1].strip(),
            flag_type=flag_type, priority=priority,
            reason=reason, language="r",
        ))

    for i, line in enumerate(lines, start=1):
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue

        # ── Load checks ────────────────────────────────────────────────────
        if _R_LOAD_FUNCS.search(line):
            load_lines.append(i)

            # encoding_not_set
            if not re.search(r"encoding|fileEncoding", line):
                _flag(i, "encoding_not_set", "Medium",
                      "read function with no encoding argument")

        # ── Join checks ────────────────────────────────────────────────────
        if _R_JOIN_FUNCS.search(line):
            merge_lines.append(i)
            if re.search(r"\bleft_join|right_join|full_join", line):
                merge_outer.append(i)

            # join_on_string — by = "column_name" that's not id-like
            m = re.search(r'by\s*=\s*["\']([^"\']+)["\']', line)
            if m:
                col = m.group(1)
                if not re.search(r"(?:id|fips|code|num|key)$", col, re.IGNORECASE):
                    _flag(i, "join_on_string", "Medium",
                          f"Join key '{col}' looks like a string column")

        # ── Column / value checks ──────────────────────────────────────────
        # zip_as_numeric
        if re.search(r"as\.numeric|as\.integer", line) and _ZIP_PATTERNS.search(line):
            _flag(i, "zip_as_numeric", "High",
                  "ZIP column cast to numeric — leading zeros will be lost")

        # total_row_risk
        if re.search(r'["\'][Tt]otal["\']', line) and re.search(r"!=|==", line):
            _flag(i, "total_row_risk", "High",
                  '"Total" row detected — verify exclusion from aggregation')

        # mean_without_median
        if re.search(r"\bmean\s*\(", line):
            nearby = lines[max(0,i-4): i+4]
            if not any(re.search(r"\bmedian\s*\(", l) for l in nearby):
                _flag(i, "mean_without_median", "Medium",
                      "mean() used — no median() nearby")

        # no_value_range_check
        if re.search(r"\b(?:mean|sum)\s*\(", line):
            nearby = lines[max(0,i-6): i+6]
            if not any(re.search(r"\b(?:min|max|range|summary)\s*\(", l) for l in nearby):
                _flag(i, "no_value_range_check", "Medium",
                      "Aggregation with no min/max/range check nearby")

        # no_category_check
        if _R_GROUP_BY.search(line):
            nearby = lines[max(0,i-6): i+6]
            if not any(re.search(r"\b(?:table|unique|levels|value_counts)\s*\(", l)
                       or "n()" in l for l in nearby):
                _flag(i, "no_category_check", "Medium",
                      "group_by() with no table()/unique() check on categories")

        # sentinel_value_risk
        m = re.search(r"!=\s*(-99+|-999+|9999)", line)
        if m:
            _flag(i, "sentinel_value_risk", "High",
                  f"Sentinel value {m.group(1)} — verify it represents missing data")

        # hardcoded_threshold
        if re.search(r"p\.value\s*[<>]=?\s*0\.05|alpha\s*=\s*0\.05", line):
            _flag(i, "hardcoded_threshold", "High",
                  "Hardcoded alpha 0.05 — document the significance threshold")

        # percentage_without_base
        if re.search(r"[*/]\s*100", line) and re.search(r"pct|percent|rate", line, re.IGNORECASE):
            nearby = lines[max(0,i-3): i+3]
            if not any(re.search(r"n=|print|cat\(", l) for l in nearby):
                _flag(i, "percentage_without_base", "High",
                      "Percentage with no denominator printed")

        # no_null_before_aggregation
        if _R_AGG_FUNCS.search(line):
            nearby = lines[max(0,i-6): i+6]
            if not any(re.search(r"na\.rm|complete\.cases|is\.na|drop_na|na\.omit", l)
                       for l in nearby):
                _flag(i, "no_null_before_aggregation", "High",
                      "Aggregation with no NA handling (na.rm / na.omit / drop_na)")

        # magic_number
        m = re.search(r"[><!]=?\s*(\d+\.\d+)\b", line)
        if m:
            val = float(m.group(1))
            if val not in (0.0, 0.5, 1.0, 100.0, 0.05) and "#" not in line:
                _flag(i, "magic_number", "Medium",
                      f"Unexplained threshold {m.group(1)} — add a comment")

    # ── Post-pass checks ────────────────────────────────────────────────────
    for lineno in load_lines:
        if not _r_has_nearby(lines, lineno, ["nrow", "dim", "str(", "length("]):
            _flag(lineno, "no_shape_check", "High",
                  "Data loaded — no nrow()/dim() check nearby")
        if not _r_has_nearby(lines, lineno, ["is.na", "complete.cases", "na.omit",
                                              "drop_na", "summary("]):
            _flag(lineno, "no_na_check", "High",
                  "Data loaded — no is.na() / complete.cases() check nearby")
        if not _r_has_nearby(lines, lineno, ["str(", "class(", "glimpse", "summary("]):
            _flag(lineno, "no_dtype_check", "Medium",
                  "Data loaded — no str()/class() dtype check nearby")

    for lineno in merge_lines:
        if not _r_has_nearby(lines, lineno, ["nrow", "dim", "print"]):
            _flag(lineno, "no_join_count_check", "High",
                  "Join with no row count check before or after")

    for lineno in merge_outer:
        if not _r_has_nearby(lines, lineno, ["anti_join", "is.na", "filter"]):
            _flag(lineno, "no_unmatched_check", "High",
                  "Outer join with no anti-join / unmatched-row check")

    flags.sort(key=lambda f: (f.line, PRIORITY_ORDER.get(f.priority, 9)))
    return flags


## Main entry point — flag_code()

In [5]:
def flag_code(path: str | Path) -> list[CodeFlag]:
    """
    Static-analyze a Python or R script and return a list of CodeFlags.
    path — absolute or relative path to the file.
    """
    path = Path(path)
    source = path.read_text(encoding="utf-8", errors="replace")
    suffix = path.suffix.lower()

    if suffix == ".py":
        try:
            tree = ast.parse(source)
        except SyntaxError as e:
            print(f"[Layer 2] SyntaxError in {path}: {e}")
            return []
        flagger = PythonFlagger(source)
        flagger.visit(tree)
        flagger.run_post_checks()
        flags = flagger.flags

    elif suffix == ".r":
        flags = flag_r(source)

    else:
        print(f"[Layer 2] Unsupported file type: {suffix}")
        return []

    flags.sort(key=lambda f: (f.line, PRIORITY_ORDER.get(f.priority, 9)))
    return flags


## Rendering — annotated source view

In [6]:
def render_flags(flags: list[CodeFlag], source_path: str | Path) -> HTML:
    """Render flagged source as annotated HTML table."""
    path = Path(source_path)
    source_lines = path.read_text(encoding="utf-8", errors="replace").splitlines()

    # Build a map: line_number -> list of flags
    flag_map: dict[int, list[CodeFlag]] = {}
    for f in flags:
        flag_map.setdefault(f.line, []).append(f)

    # Legend
    seen_types = sorted({f.flag_type for f in flags})
    legend_items = "".join(
        f'<span style="background:{FLAG_COLORS.get(ft,"#eee")};'
        f'padding:2px 8px;margin:2px;border-radius:3px;font-size:0.85em;">'
        f'{ft}</span>'
        for ft in seen_types
    )

    # Build summary table
    summary_rows = ""
    for ft in seen_types:
        ft_flags = [f for f in flags if f.flag_type == ft]
        color = FLAG_COLORS.get(ft, "#eee")
        lines_str = ", ".join(str(f.line) for f in ft_flags)
        priority = ft_flags[0].priority
        summary_rows += (
            f'<tr>'
            f'<td style="background:{color};padding:4px 8px;">{ft}</td>'
            f'<td style="padding:4px 8px;">{priority}</td>'
            f'<td style="padding:4px 8px;">{len(ft_flags)}</td>'
            f'<td style="padding:4px 8px;font-family:monospace;">{lines_str}</td>'
            f'</tr>'
        )

    # Build annotated source
    code_rows = ""
    for i, line in enumerate(source_lines, start=1):
        line_flags = flag_map.get(i, [])
        if line_flags:
            # Background: blend flag colors (use first High flag, else first flag)
            primary = sorted(line_flags, key=lambda f: PRIORITY_ORDER.get(f.priority, 9))[0]
            bg = FLAG_COLORS.get(primary.flag_type, "#fff9c4")
            annotation = " | ".join(
                f'<span style="background:{FLAG_COLORS.get(f.flag_type,"#eee")};'
                f'padding:1px 5px;border-radius:3px;font-size:0.8em;">'
                f'{f.flag_type}: {f.reason}</span>'
                for f in line_flags
            )
            code_rows += (
                f'<tr style="background:{bg}22;">'
                f'<td style="color:#888;padding:2px 8px;font-family:monospace;'
                f'user-select:none;min-width:40px;text-align:right;">{i}</td>'
                f'<td style="font-family:monospace;white-space:pre;padding:2px 12px;">'
                f'{line.replace("<","&lt;").replace(">","&gt;")}</td>'
                f'<td style="padding:2px 8px;">{annotation}</td>'
                f'</tr>'
            )
        else:
            code_rows += (
                f'<tr>'
                f'<td style="color:#ccc;padding:2px 8px;font-family:monospace;'
                f'user-select:none;min-width:40px;text-align:right;">{i}</td>'
                f'<td style="font-family:monospace;white-space:pre;color:#555;'
                f'padding:2px 12px;">{line.replace("<","&lt;").replace(">","&gt;")}</td>'
                f'<td></td>'
                f'</tr>'
            )

    html = f"""
    <style>
      .l2-report {{ font-family: sans-serif; max-width: 1200px; }}
      .l2-report h2 {{ font-size: 1.1em; margin-bottom: 6px; }}
      .l2-report table {{ border-collapse: collapse; width: 100%; margin-bottom: 16px; }}
      .l2-report td {{ vertical-align: top; }}
      .l2-report tr:hover {{ filter: brightness(0.95); }}
    </style>
    <div class="l2-report">
      <h2>Layer 2 — Code Risk Report: <code>{path.name}</code></h2>
      <p style="color:#555;"><strong>{len(flags)}</strong> flag(s) across <strong>{len(seen_types)}</strong> type(s)</p>
      <div style="margin-bottom:12px;">{legend_items}</div>

      <h2>Summary</h2>
      <table>
        <tr style="background:#f1f3f5;font-weight:bold;">
          <td style="padding:4px 8px;">Flag type</td>
          <td style="padding:4px 8px;">Priority</td>
          <td style="padding:4px 8px;">Count</td>
          <td style="padding:4px 8px;">Lines</td>
        </tr>
        {summary_rows}
      </table>

      <h2>Annotated source</h2>
      <table>{code_rows}</table>
    </div>
    """
    return HTML(html)


## Tests

In [7]:
# ── Unit tests ────────────────────────────────────────────────────────────────

EXAMPLES_DIR = Path("layer2_examples")
RISKY_PY = EXAMPLES_DIR / "example_risky.py"
CLEAN_PY  = EXAMPLES_DIR / "example_clean.py"
RISKY_R   = EXAMPLES_DIR / "example_risky.R"


def run_tests() -> None:
    passed = 0
    failed = 0

    def check(name: str, condition: bool, detail: str = "") -> None:
        nonlocal passed, failed
        if condition:
            print(f"  PASS  {name}")
            passed += 1
        else:
            print(f"  FAIL  {name}" + (f"  — {detail}" if detail else ""))
            failed += 1

    # ── Python risky ─────────────────────────────────────────────────────────
    print("\n── Python: example_risky.py ──")
    py_flags = flag_code(RISKY_PY)
    py_types = {f.flag_type for f in py_flags}

    for ft in ["no_shape_check", "no_na_check", "zip_as_numeric", "total_row_risk",
               "sentinel_value_risk", "no_join_count_check", "no_unmatched_check",
               "hardcoded_threshold", "no_null_before_aggregation"]:
        check(ft, ft in py_types, f"not found in {py_types}")

    for ft in ["encoding_not_set", "no_dtype_check", "mean_without_median",
               "no_value_range_check", "no_category_check", "join_on_string", "magic_number"]:
        check(ft, ft in py_types, f"not found in {py_types}")

    # ── Python clean — no High flags ─────────────────────────────────────────
    print("\n── Python: example_clean.py ──")
    clean_flags = flag_code(CLEAN_PY)
    clean_high = [f for f in clean_flags if f.priority == "High"]
    check("clean script: no High flags", len(clean_high) == 0,
          f"got {len(clean_high)}: " + str([f.flag_type for f in clean_high]))

    # ── R risky ──────────────────────────────────────────────────────────────
    print("\n── R: example_risky.R ──")
    r_flags = flag_code(RISKY_R)
    r_types = {f.flag_type for f in r_flags}

    for ft in ["no_shape_check", "no_na_check", "zip_as_numeric", "total_row_risk",
               "sentinel_value_risk", "no_join_count_check", "no_unmatched_check",
               "hardcoded_threshold", "no_null_before_aggregation",
               "encoding_not_set", "join_on_string", "mean_without_median"]:
        check(ft, ft in r_types, f"not found in {r_types}")

    total = passed + failed
    print(f"\n{'─'*40}")
    print(f"{passed}/{total} tests passed", "✓" if failed == 0 else f"({failed} failed)")


run_tests()



── Python: example_risky.py ──
  PASS  no_shape_check
  PASS  no_na_check
  PASS  zip_as_numeric
  PASS  total_row_risk
  PASS  sentinel_value_risk
  PASS  no_join_count_check
  PASS  no_unmatched_check
  PASS  hardcoded_threshold
  PASS  no_null_before_aggregation
  PASS  encoding_not_set
  PASS  no_dtype_check
  PASS  mean_without_median
  PASS  no_value_range_check
  PASS  no_category_check
  PASS  join_on_string
  PASS  magic_number

── Python: example_clean.py ──
  PASS  clean script: no High flags

── R: example_risky.R ──
  PASS  no_shape_check
  PASS  no_na_check
  PASS  zip_as_numeric
  PASS  total_row_risk
  PASS  sentinel_value_risk
  PASS  no_join_count_check
  PASS  no_unmatched_check
  PASS  hardcoded_threshold
  PASS  no_null_before_aggregation
  PASS  encoding_not_set
  PASS  join_on_string
  PASS  mean_without_median

────────────────────────────────────────
29/29 tests passed ✓


## Run on example scripts

In [8]:
# ── Python risky example ─────────────────────────────────────────────────────
py_flags = flag_code(RISKY_PY)
display(render_flags(py_flags, RISKY_PY))


Flag type,Priority,Count,Lines
encoding_not_set,Medium,2,"8, 14"
hardcoded_threshold,High,1,50
join_on_string,Medium,2,"39, 42"
magic_number,Medium,1,33
mean_without_median,Medium,2,"26, 36"
no_category_check,Medium,3,"23, 30, 61"
no_dtype_check,Medium,2,"8, 14"
no_join_count_check,High,3,"39, 42, 45"
no_na_check,High,2,"8, 14"
no_null_before_aggregation,High,7,"23, 26, 30, 36, 54, 58, 61"


In [9]:
# ── R risky example ───────────────────────────────────────────────────────────
r_flags = flag_code(RISKY_R)
display(render_flags(r_flags, RISKY_R))


Flag type,Priority,Count,Lines
encoding_not_set,Medium,2,"7, 13"
hardcoded_threshold,High,1,53
join_on_string,Medium,2,"43, 46"
magic_number,Medium,1,35
mean_without_median,Medium,2,"28, 40"
no_category_check,Medium,1,24
no_dtype_check,Medium,2,"7, 13"
no_join_count_check,High,2,"43, 46"
no_na_check,High,2,"7, 13"
no_null_before_aggregation,High,4,"25, 28, 32, 40"


In [10]:
# ── Clean Python example — should be minimal ──────────────────────────────────
clean_flags = flag_code(CLEAN_PY)
print(f"{len(clean_flags)} flag(s) on clean script")
if clean_flags:
    display(render_flags(clean_flags, CLEAN_PY))
else:
    print("No flags. Clean script passed.")


5 flag(s) on clean script


## Analyze your own script

Replace the path below with any `.py` or `.R` file.

## Repo scanner — scan a directory of scripts

In [11]:
def scan_repo(repo_path: str | Path,
              extensions: tuple[str, ...] = (".py", ".r"),
              exclude_dirs: set[str] | None = None) -> dict[str, list[CodeFlag]]:
    """
    Recursively scan a directory for Python and R scripts and flag each one.

    Returns a dict: file_path_str -> list[CodeFlag].
    Only includes files that produced at least one flag.
    """
    repo = Path(repo_path)
    exclude = exclude_dirs or {".git", "__pycache__", ".venv", "venv", "node_modules",
                               ".tox", "dist", "build", ".mypy_cache"}
    results: dict[str, list[CodeFlag]] = {}

    for path in sorted(repo.rglob("*")):
        if not path.is_file():
            continue
        if any(part in exclude for part in path.parts):
            continue
        if path.suffix.lower() not in extensions:
            continue
        flags = flag_code(path)
        if flags:
            results[str(path)] = flags

    return results


def render_repo_summary(results: dict[str, list[CodeFlag]], repo_path: str | Path) -> HTML:
    """
    Render a repo-level summary table: one row per file, flag counts by priority.
    """
    repo = Path(repo_path)
    if not results:
        return HTML("<p>No flags found in repo.</p>")

    # Count High and Medium per file
    rows = ""
    total_high = total_medium = 0
    for fpath, flags in sorted(results.items(), key=lambda kv: -len([f for f in kv[1] if f.priority=="High"])):
        rel = Path(fpath).relative_to(repo)
        high = [f for f in flags if f.priority == "High"]
        med  = [f for f in flags if f.priority == "Medium"]
        total_high += len(high)
        total_medium += len(med)

        # Unique flag types
        types = sorted({f.flag_type for f in flags})
        badges = "".join(
            f'<span style="background:{FLAG_COLORS.get(t,"#eee")};'
            f'padding:1px 6px;margin:1px;border-radius:3px;font-size:0.78em;">{t}</span>'
            for t in types
        )
        high_cell = f'<td style="padding:4px 8px;color:#e03131;font-weight:bold;">{len(high)}</td>' if high else '<td style="padding:4px 8px;color:#aaa;">—</td>'
        rows += (
            f'<tr>'
            f'<td style="padding:4px 8px;font-family:monospace;">{rel}</td>'
            f'{high_cell}'
            f'<td style="padding:4px 8px;">{len(med)}</td>'
            f'<td style="padding:4px 8px;">{badges}</td>'
            f'</tr>'
        )

    html = f"""
    <style>
      .repo-report {{ font-family: sans-serif; max-width: 1200px; }}
      .repo-report table {{ border-collapse: collapse; width: 100%; }}
      .repo-report td {{ vertical-align: top; border-bottom: 1px solid #eee; }}
      .repo-report tr:hover {{ background: #f8f9fa; }}
    </style>
    <div class="repo-report">
      <h2>Layer 2 — Repo Scan: <code>{repo.name}/</code></h2>
      <p><strong>{len(results)}</strong> files flagged &nbsp;|&nbsp;
         <strong style="color:#e03131;">{total_high} High</strong> &nbsp;|&nbsp;
         <strong>{total_medium} Medium</strong></p>
      <table>
        <tr style="background:#f1f3f5;font-weight:bold;">
          <td style="padding:4px 8px;">File</td>
          <td style="padding:4px 8px;">High</td>
          <td style="padding:4px 8px;">Medium</td>
          <td style="padding:4px 8px;">Flag types</td>
        </tr>
        {rows}
      </table>
    </div>
    """
    return HTML(html)


In [12]:
# Point to a repo and scan all .py and .R files.
# Change this path to scan any project directory.

REPO_PATH = Path("/Users/akastanis/Git_work/national-flood-insurance")   # swap for any absolute path

repo_results = scan_repo(REPO_PATH)
display(render_repo_summary(repo_results, REPO_PATH))


File,High,Medium,Flag types
etl/etl_2024.R,5,7,encoding_not_setno_category_checkno_dtype_checkno_join_count_checkno_na_checkno_shape_checkno_value_range_check
test.R,2,3,mean_without_medianno_category_checkno_null_before_aggregationno_value_range_check


## Decision point detector

Surfaces methodological choices that affect the story outcome —
things an editor or peer reviewer should explicitly sign off on.

These are **not** bugs. They are places where the analyst made a judgment call
that a second person should understand and verify.

In [13]:
from dataclasses import dataclass as _dc

@_dc
class DecisionPoint:
    line: int
    code: str
    category: str   # e.g. "filter_threshold", "groupby_key", "join_type"
    question: str   # the question a reviewer should answer


# ── Decision point patterns (Python) ────────────────────────────────────────

_DP_PATTERNS = [
    # Filter threshold — any comparison with a non-trivial numeric value
    ("filter_threshold",
     re.compile(r"(?:df|data|gdf)\[.*?[><!]=?\s*\d+(?:\.\d+)?\b"),
     "What is the basis for this filter threshold? Is it from the data definition, "
     "a legal standard, or an analytic choice?"),

    # Date cutoff — hard-coded date or year in a filter
    ("date_cutoff",
     re.compile(r"[><!]=?\s*['\"](?:19|20)\d{2}|\.dt\.|pd\.to_datetime"),
     "Why this date boundary? Was it chosen before or after seeing the data?"),

    # Groupby key — what unit of analysis was chosen
    ("unit_of_analysis",
     re.compile(r"\.groupby\(['\"][\w_]+['\"]"),
     "This groupby key defines the unit of analysis. Was the right level of aggregation chosen?"),

    # Join type — inner vs left vs outer changes who's in the denominator
    ("join_type",
     re.compile(r"how=['\"](?:left|right|outer|inner)['\"]"),
     "Join type affects which records appear in the result. "
     "Was the right join type chosen? Who is excluded by an inner join?"),

    # Statistical test choice
    ("stat_test_choice",
     re.compile(r"\b(?:ttest_ind|ttest_1samp|mannwhitneyu|chi2_contingency|anova|pearsonr|spearmanr)\s*\("),
     "Which statistical test was chosen and why? Were the assumptions (normality, independence) checked?"),

    # Exclusion filter — rows dropped from the analysis
    ("exclusion_filter",
     re.compile(r"(?:df|data)\[(?:df|data)\[.*?\]\s*(?:!=|>|<|>=|<=|~)"),
     "This filter removes rows from the analysis. Are excluded rows documented? "
     "What share of the data do they represent?"),

    # Column selection — specific columns chosen for analysis
    ("column_selection",
     re.compile(r"\[\[[\w'\",\s]+\]\]"),
     "These columns were selected for analysis. Were alternative columns considered? "
     "Is this the right metric for the question?"),

    # Rate denominator — population used to normalize
    ("rate_denominator",
     re.compile(r"/\s*(?:df|data|pop|total|n)\b"),
     "What population is this rate normalized against? Is the denominator consistent "
     "across comparisons?"),

    # Time period — year or range selection
    ("time_period",
     re.compile(r"year\s*==\s*\d{4}|(?:19|20)\d{2}|\.dt\.year"),
     "Which time period was chosen? Why this year or range? "
     "Were trend patterns checked before and after?"),

    # Deduplication choice
    ("deduplication",
     re.compile(r"\.drop_duplicates\(|\.duplicated\("),
     "Records were deduplicated. What was the key? Which duplicate was kept? "
     "How many records were removed?"),
]


def find_decision_points(source: str, language: str = "python") -> list[DecisionPoint]:
    """
    Scan source code for methodological decision points — places where an
    analyst made a judgment call that affects the story outcome.
    """
    lines = source.splitlines()
    points: list[DecisionPoint] = []
    seen: set[tuple[int, str]] = set()  # deduplicate same line + category

    for i, line in enumerate(lines, start=1):
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        c = _code_only(line)

        for category, pattern, question in _DP_PATTERNS:
            if pattern.search(c):
                key = (i, category)
                if key not in seen:
                    seen.add(key)
                    points.append(DecisionPoint(
                        line=i,
                        code=stripped[:120],
                        category=category,
                        question=question,
                    ))

    return sorted(points, key=lambda p: p.line)


def render_decision_points(points: list[DecisionPoint], source_path: str | Path) -> HTML:
    """Render decision points as a review checklist."""
    path = Path(source_path)
    if not points:
        return HTML("<p>No decision points detected.</p>")

    CATEGORY_COLORS = {
        "filter_threshold":   "#ff922b",
        "date_cutoff":        "#ffd43b",
        "unit_of_analysis":   "#74c0fc",
        "join_type":          "#ff6b6b",
        "stat_test_choice":   "#63e6be",
        "exclusion_filter":   "#ff922b",
        "column_selection":   "#dee2e6",
        "rate_denominator":   "#f03e3e",
        "time_period":        "#ffa8a8",
        "deduplication":      "#a9e34b",
    }

    rows = ""
    for p in points:
        color = CATEGORY_COLORS.get(p.category, "#eee")
        rows += (
            f'<tr>'
            f'<td style="padding:6px 8px;color:#888;font-family:monospace;text-align:right;">{p.line}</td>'
            f'<td style="padding:6px 8px;">'
            f'  <span style="background:{color};padding:2px 7px;border-radius:3px;'
            f'  font-size:0.82em;margin-right:6px;">{p.category}</span>'
            f'</td>'
            f'<td style="padding:6px 8px;font-family:monospace;font-size:0.88em;">'
            f'  {p.code.replace("<","&lt;").replace(">","&gt;")}</td>'
            f'<td style="padding:6px 8px;font-size:0.88em;color:#444;">{p.question}</td>'
            f'</tr>'
        )

    html = f"""
    <style>
      .dp-report {{ font-family: sans-serif; max-width: 1300px; }}
      .dp-report table {{ border-collapse: collapse; width: 100%; }}
      .dp-report td {{ vertical-align: top; border-bottom: 1px solid #eee; }}
      .dp-report tr:hover {{ background: #f8f9fa; }}
    </style>
    <div class="dp-report">
      <h2>Decision Points: <code>{path.name}</code></h2>
      <p style="color:#555;">{len(points)} methodological choice(s) to verify with an editor or peer reviewer</p>
      <table>
        <tr style="background:#f1f3f5;font-weight:bold;">
          <td style="padding:6px 8px;min-width:40px;">Line</td>
          <td style="padding:6px 8px;">Category</td>
          <td style="padding:6px 8px;">Code</td>
          <td style="padding:6px 8px;">Question for reviewer</td>
        </tr>
        {rows}
      </table>
    </div>
    """
    return HTML(html)


In [14]:
# ── Decision points on the risky example ──────────────────────────────────────
risky_src = RISKY_PY.read_text()
dp = find_decision_points(risky_src)
display(render_decision_points(dp, RISKY_PY))


Line,Category,Code,Question for reviewer
23,unit_of_analysis,"total = df[df[""county""] != ""Total""].groupby(""county"")[""count""].sum()",This groupby key defines the unit of analysis. Was the right level of aggregation chosen?
23,exclusion_filter,"total = df[df[""county""] != ""Total""].groupby(""county"")[""count""].sum()",This filter removes rows from the analysis. Are excluded rows documented? What share of the data do they represent?
30,unit_of_analysis,"by_county = df.groupby(""county"")[""evictions""].sum()",This groupby key defines the unit of analysis. Was the right level of aggregation chosen?
33,filter_threshold,"high_risk = df[df[""rate""] > 0.15]","What is the basis for this filter threshold? Is it from the data definition, a legal standard, or an analytic choice?"
33,exclusion_filter,"high_risk = df[df[""rate""] > 0.15]",This filter removes rows from the analysis. Are excluded rows documented? What share of the data do they represent?
36,exclusion_filter,"income_avg = df[df[""income""] != -99][""income""].mean()",This filter removes rows from the analysis. Are excluded rows documented? What share of the data do they represent?
45,join_type,"left = df.merge(df2, on=""id"", how=""left"")",Join type affects which records appear in the result. Was the right join type chosen? Who is excluded by an inner join?
49,stat_test_choice,"t, p = stats.ttest_ind(df[""group_a""], df[""group_b""])","Which statistical test was chosen and why? Were the assumptions (normality, independence) checked?"
61,unit_of_analysis,"annual = df.groupby(""year"")[""evictions""].sum().pct_change()",This groupby key defines the unit of analysis. Was the right level of aggregation chosen?


## Scan any repo or file

```python
# Scan a full project directory
results = scan_repo("/path/to/your/analysis/repo")
display(render_repo_summary(results, "/path/to/your/analysis/repo"))

# Drill into a specific file
flags = flag_code("/path/to/your/analysis/repo/analysis.py")
display(render_flags(flags, "/path/to/your/analysis/repo/analysis.py"))

# Decision points on a file
source = Path("/path/to/analysis.py").read_text()
dp = find_decision_points(source)
display(render_decision_points(dp, "/path/to/analysis.py"))
```

In [15]:
# path = Path("/path/to/your/analysis.py")
# flags = flag_code(path)
# display(render_flags(flags, path))
